# German Credit SHAP-based Scorecard with RandomForest (scorecardpl)

This notebook builds a SHAP-derived scorecard using a RandomForest classifier trained on WOE features from the German Credit dataset.

Steps:
- Load dataset and map target to 0/1
- Train/valid split + variable filtering
- WOE/IV binning (numeric + categorical)
- Train RandomForest on WOE features
- Build scorecard points via SHAP contributions (log-odds scale)
- Evaluate AUC/KS, compute PSI, and export a simple report

Note: Compatible with SHAP 0.48+; the library normalizes attribution shapes and falls back from TreeExplainer when needed.

In [ ]:
%matplotlib inline
# If needed: !pip -q install shap

import os
import numpy as np
import pandas as pd
import polars as pl
from sklearn.ensemble import RandomForestClassifier

from scorecardpl import (
    split_df, var_filter, woebin, woebin_ply,
    scorecard_ply, perf_eva, iv_summary, psi,
    bins_export_json, bins_import_json, write_method_report,
    scorecard_shap, SHAPScorecardModel, woebin_plot,
)

os.makedirs('plots', exist_ok=True)


## Load German Credit data
The label maps to 0/1 as: 1=good (0), 2=bad (1).

In [ ]:
uci_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data'
cols = [
    'Status', 'Duration', 'CreditHistory', 'Purpose', 'CreditAmount',
    'Savings', 'Employment', 'InstallmentRate', 'PersonalStatusSex', 'OtherDebtors',
    'ResidenceSince', 'Property', 'Age', 'OtherInstallmentPlans', 'Housing',
    'ExistingCredits', 'Job', 'Liables', 'Telephone', 'ForeignWorker', 'class'
]
df_pd = pd.read_csv(uci_url, sep=' ', header=None, names=cols)
df_pd['y'] = (df_pd['class'] == 2).astype(int)
df_pd = df_pd.drop(columns=['class'])
df_pl = pl.from_pandas(df_pd)
df_pl.head()


## Split, filter, and bin

In [ ]:
y = 'y'
xs = [c for c in df_pl.columns if c != y]
train, valid = split_df(df_pl, y=y, test_size=0.3, random_state=42)
train = var_filter(train, y=y, x=xs)
bins = woebin(
    train, y=y, x=[c for c in train.columns if c != y],
    bins=6, method='chi2', chi2_params={'init_bins': 60},
    monotonic='auto', cat_max_bins=5,
)
ivsum = iv_summary(bins)
ivsum


## Train RandomForest and build SHAP scorecard

In [ ]:
est = RandomForestClassifier(n_estimators=200, max_depth=3, random_state=42)
sc_shap = scorecard_shap(
    bins=bins, y=y, data=train, estimator=est,
    shap_sample_n=2000, pdo=20.0, base_score=600.0, odds=50.0
)
type(sc_shap)


## Evaluate and score

In [ ]:
proba = sc_shap.predict_proba(valid, bins)
perf = perf_eva(valid[y], proba, plot='both', save_prefix='plots/german_credit_rf_shap')
scores = sc_shap.predict_points(valid, bins)
perf, scores.head(10)
